# ML for classification
## Churn project


##### Empresa telefonica A con clientes/consumidores que consumen los servicios de la empresa (TV, internet, telefono, etc). Algunos pueden no estar felices con los servicios, por lo que querrán irse a otra empres telefonica B. 

##### Hay que identificar que clientes se van a ir de la empresa A a la empresa B, y todo esto a partir de probabilidades (probabilidad de _churn_)

##### Para analizar esto, se necesita una **clasificación binaria**, que es un método de aprendizaje supervisado. 

$$ g(x_{i}) \approx y_{i} $$

##### Donde $y_{i} \in {0,1} $, siendo $1$ un ejemplo  (ya sea _churn_ o _spam_, etc) y $0$ un ejemplo negativo (es decir, _no-churn_, _no-spam_, etc). Por otro lado, $g_{i}$ tal que $g_{i} \in (0,1)$ representa de que el cliente $i-th$ _churnee_

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Validación de los datos

In [4]:
df=pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [7]:
df.columns=df.columns.str.lower().str.replace(" ","_")
categorical_columns=list(df.dtypes[df.dtypes=="object"].index)
for c in categorical_columns:
    df[c]=df[c].str.lower().str.replace(" ","_")
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [ ]:
df.dtypes #totalcharges debería ser un número y seniorcitizen un objeto

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

In [9]:
tc=pd.to_numeric(df.totalcharges,errors='coerce') #el 'coerce' hace que se ignoren los errores

In [14]:
df.totalcharges=pd.to_numeric(df.totalcharges,errors='coerce')

In [15]:
df.totalcharges=df.totalcharges.fillna(0)

In [18]:
df.totalcharges.isnull().sum()

np.int64(0)

In [20]:
#en cuanto a la variable "churn", estamos intresados no en "yes" or "no" si no más bien en unos (1) y ceros (0)
df.churn=(df.churn=="yes").astype(int)
df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

## Configurando el entorno de validación

Se hará lo mismo que se hizo en la sección anterior (la de regresión), pero ahora, en vez de hacerlo de forma manual, se va a hacer a través de _sickit-learn_ (los indices ya van a venir aleatorios)

In [21]:
from sklearn.model_selection import train_test_split

In [27]:
df_full_train,df_test=train_test_split(df,test_size=0.2,random_state=1)
len(df_full_train),len(df_test)

(5634, 1409)

In [29]:
df_train,df_val=train_test_split(df_full_train,test_size=0.25,random_state=1)
len(df_train),len(df_val),len(df_test)

(4225, 1409, 1409)

In [30]:
df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

In [31]:
y_train=df_train.churn.values
y_val=df_val.churn.values
y_test=df_test.churn.values

In [32]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

## Análisis exploratorio de datos

Comprobar valores faltantes
Observar la variable objetivo
Observar variables numericas y categoricas

In [34]:
df_full_train.isnull().sum() #no hay valores faltantes

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [36]:
df_full_train.churn.value_counts()

churn
0    4113
1    1521
Name: count, dtype: int64

In [38]:
df_full_train.churn.value_counts(normalize=True) #la proporción de unos (1) es la churn rate (tasa de abandono)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [42]:
global_churn_rate=df_full_train.churn.mean() #da lo mismo que el churn rate anterior
round(global_churn_rate,2)

np.float64(0.27)

In [43]:
df_full_train.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

In [44]:
#estamos interesados en 'tenure', 'monthlycharges' y 'totalcharges'
numeric=['tenure','monthlycharges','totalcharges']

In [45]:
df_full_train.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines',
       'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv',
       'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

In [46]:
categorical=['gender', 'seniorcitizen', 'partner', 'dependents', 'phoneservice', 'multiplelines',
       'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv',
       'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'churn']

In [47]:
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
churn               2
dtype: int64

## Importancia de las características: tasa de abandono y ratio de riesgo

¿Cuáles atributos afectan a nuestra variable objetivo?

### Churn rate

In [54]:
#churn rate en diferentes grupos
churn_female=df_full_train[df_full_train.gender=='female'].churn.mean()
churn_male=df_full_train[df_full_train.gender=='male'].churn.mean()
print(churn_female)
print(churn_male)
print(global_churn_rate)

0.27682403433476394
0.2632135306553911
0.26996805111821087


In [55]:
with_partner=df_full_train[df_full_train.partner=='yes'].churn.mean()
without_partner=df_full_train[df_full_train.partner=="no"].churn.mean()
print(with_partner)
print(without_partner)

0.20503330866025166
0.3298090040927694


La **diferencia** entre ambas tasas de abandono considerando la variable 'partner' es significativa. Si la tasa global es mayor que la particular, hay menos probabilidades de abandono. Lo contrario sucede cuando la tasa global es menor a la particular.

Por otro lado, el cociente entre ambas (la particular / la global), es una **tasa de riesgo**. Si el ratio es mayor a 1, hay mas chances de churnear, y si es menor a 1, se da el caso contrario.

La **diferencia** y el **riesgo** son muy similares, es decir, nos dan la misma información, pero de formas distintas. Con la diferencia no vemos que tan grande es una tasa de churn respecto de la otra, en cambio con el riesgo si, es decir, lo expresamos en términos relativos

In [ ]:
with_partner/global_churn_rate #la tasa es 25% mas chica que la global

np.float64(0.7594724924338315)

In [58]:
without_partner/global_churn_rate #la tasa es 22% mas grande que la global

np.float64(1.2216593879412643)

In [62]:
df_group=df_full_train.groupby('gender').churn.agg(['mean','count'])
df_group['diff']=df_group['mean']-global_churn_rate
df_group['risk']=df_group['mean']/global_churn_rate
df_group

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


In [65]:
from IPython.display import display

In [66]:
#queremos hacer lo anterior para todas las variables categoricas
for c in categorical:
    print(c)
    df_group=df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group['diff']=df_group['mean']-global_churn_rate
    df_group['risk']=df_group['mean']/global_churn_rate
    display(df_group)

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208


partner


...

,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560


paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


churn


,mean,count,diff,risk
churn,,,,
0,0.0,4113,-0.269968,0.000000
1,1.0,1521,0.730032,3.704142
